In [4]:
import yt_dlp
import os
import cv2
from ultralytics import YOLO
import csv

Prompt user to select a file from their system or download a VOD from youtube

In [2]:
def get_user_choice():

    while True:
        choice = input("Would you like to provide a (1) File Path or (2) YouTube URL Link? Enter 1 or 2: ").strip()

        if choice in ["1", "2"]:
            return choice
        print("Invalid choice. Please enter 1 for File Path or 2 for URL Link.")

def get_file_or_link(choice):

    if choice == "1":
        upload_type = "file"
        user_input = ""

    elif choice == "2":
        upload_type = "url"
        user_input = input("Enter the URL link: ").strip()

    return (upload_type, user_input)

# Prompt user for choice
user_choice = get_user_choice()
video_type, user_input = get_file_or_link(user_choice)

print(f"You selected a {video_type.upper()} with input: {user_input}")

if video_type == "url":  # If user selected URL, download the video from YouTube
    save_folder = "VODS"
    os.makedirs(save_folder, exist_ok=True)

    ydl_opts = {
        "format": "bestvideo[height=720]",
        "outtmpl": f"{save_folder}/%(title)s.%(ext)s",
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(user_input, download=True)
        video_title = info_dict.get('title', 'unknown_title').replace(" ", "_")
        video_ext = info_dict.get('ext', 'mp4')

    VOD = os.path.abspath(os.path.join(save_folder, f"{video_title}.{video_ext}"))

elif video_type == "file":

    file_path = input("Paste the full path to your video file: ").strip()

    VOD = os.path.abspath(file_path)
    print("Selected file:", VOD)

# Print the final video path
print(f"\n✅ Final video path: {VOD}")


You selected a FILE with input: 
Selected file: C:\Users\ianmk\CODING\val_vision\VODS\test.webm

✅ Final video path: C:\Users\ianmk\CODING\val_vision\VODS\test.webm


Now we have a variable "VOD" which is a string that represents the system path to the VOD

From here we will

1) Apply the classification model to remove all extra frames (Casting, timeouts, pre-round, etc...)

2) Apply the detection model to output bounding box coords and frame ID

3) Pass the coords and frame ID to OCR model and output data

In [5]:
# Load the trained YOLO classification model
classify_model = YOLO("runs/classify/train/weights/best.pt")
killfeed_box_model = YOLO("killfeed/results/run21/weights/best.pt")

cap = cv2.VideoCapture(VOD)  # Load the video

batch_size = 10  # Number of frames to process at once
frame_buffer = []
frame_count = 0

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the killfeed region (x, y, width, height.) 
# fThis is the area our model will check for killfeed boxes.
killfeed_x, killfeed_y, killfeed_w, killfeed_h = 850, 50, 430, 350

# Initialize a list to store killfeed box coordinates
killfeed_boxes = []

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break  # Exit when no more frames

    frame_buffer.append(frame)
    frame_count += 1

    if len(frame_buffer) == batch_size:
        # Pass batch of frames to your models
        for f in frame_buffer:
            results = classify_model(frame)
            prediction = results[0].probs.top1 # prediction = 1 if current frame has gameplay, 0 otherwise
            #print("Prediction:", prediction)

            if prediction == 1:
                # NOW WE WANT THE FRAME TO BE PASSED TO THE OBJECT DETECTION MODEL
                # If the detection box is within our killfeed region, we want to pass the frame to the killfeed model
                killfeed_results = killfeed_box_model(f)

                # Step 3: Check if detected objects are inside the killfeed region
                for result in killfeed_results:
                    for box in result.boxes:
                        x_min, y_min, x_max, y_max = box.xyxy[0]  # Get bounding box coordinates

                        # Check if the box is within the killfeed region
                        if (
                            killfeed_x <= x_min <= killfeed_x + killfeed_w and
                            killfeed_y <= y_min <= killfeed_y + killfeed_h
                        ):
                            print(f"Killfeed detected at: {x_min}, {y_min}, {x_max}, {y_max}")
                            
                            # Save the box coordinates to the list
                            killfeed_boxes.append([x_min, y_min, x_max, y_max])
                            

        # Clear buffer after processing
        frame_buffer.clear()

# After all processing is done (or at any checkpoint), write to CSV
with open("killfeed_boxes.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["x_min", "y_min", "x_max", "y_max"])  # Header
    writer.writerows(killfeed_boxes)


# Process remaining frames from video 
if frame_buffer:
    for f in frame_buffer:
        pass  # Replace this with model processing logic

cap.release()
cv2.destroyAllWindows()





0: 640x640 0 0.96, 1 0.04, 66.0ms
Speed: 25.0ms preprocess, 66.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 64.0ms
Speed: 20.0ms preprocess, 64.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 117.0ms
Speed: 21.0ms preprocess, 117.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 58.0ms
Speed: 22.0ms preprocess, 58.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 57.0ms
Speed: 18.0ms preprocess, 57.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 60.0ms
Speed: 23.0ms preprocess, 60.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 63.0ms
Speed: 17.0ms preprocess, 63.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 0 0.96, 1 0.04, 56.0ms
Speed: 19.0ms preprocess, 56.0ms